# Serving MultiSwitch with vLLM

**Duration:** ~35-45 min (first run; composing the checkpoint is most of it)

The MultiSwitch notebooks so far read routing decisions straight out of a HuggingFace model. This one puts a `multi` checkpoint behind a **vLLM server** and drives it over HTTP - including the case the other notebooks cannot reach: several requests, naming different adapters, batched together by the scheduler.

*Why `/v1/completions` and not mellea:* mellea's `OpenAIBackend` invokes one adapter per call, and `/v1/chat/completions` applies the chat template server-side, which binds a single `adapter_name`. A prompt carrying two control tokens has to be built client-side and sent as token ids on the completions endpoint - the same path `tests/integration/test_multi_switch_serving_e2e.py` uses. Building it client-side does **not** mean skipping the chat template: section 4 still renders through it and then splices, so the prompt keeps its role markers and generation prompt.

**What you'll learn:**
- How to serve a `switch_type="multi"` checkpoint (no special vLLM flag - the engine is chosen from `config.json`)
- How to send a multi-adapter prompt over HTTP as token ids, since neither the chat template nor mellea can express one
- How to check that batching does not leak between requests, by comparing isolated and concurrent responses
- Which serving flags actually matter here, and why `max_num_seqs` is the one people get wrong

**Adapters used:** the LoRA flavors of [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) and [Core](https://huggingface.co/ibm-granite/granitelib-core-r1.0). As in [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb), what is under test is the routing and the serving path rather than any adapter's task output.

## Prerequisites

1. **Install** the vLLM and compose extras:

In [ ]:
%pip install "granite-switch[vllm,compose]"

2. **Hugging Face login:**

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

3. **A GPU is required** - vLLM will not start without one. A single T4 is enough for `granite-4.1-3b` at the reduced `--max-model-len` used below.
4. **Disk and bandwidth.** ~20 GB free for the base model plus adapters plus the composed checkpoint.
5. **If you have already run [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb)**, point `MODEL_OUT` at the checkpoint it produced and skip section 2 entirely.

Full environment notes are in [`../PREREQUISITES.md`](../PREREQUISITES.md).

---

## What changes when you serve it

Nothing about the switch. The engine is baked into the checkpoint, so vLLM picks it up from `config.json`:

```
config.json                      vllm/switch/__init__.py
  "switch_type": "multi"   -->     create_switch(config)
                                     switch_type == "multi"
                                       -> MultiSwitch
                                     else
                                       -> SingleSwitch
```

What *does* change is who owns the bookkeeping. Under HuggingFace you hand the model one padded batch. Under vLLM the scheduler decides what runs together, and requests from different users land in the same forward pass:

```
  forward pass N
  +-----------------------------------------------+
  | req A rows:  <|core|>  ...   routing -> 1     |   positions restart at 0
  | req B rows:  <|guard|> ...   routing -> 2     |   per request, so each one
  | req C rows:  (no control)    routing -> 0     |   counts its own tokens
  +-----------------------------------------------+
```

Per-request isolation comes from three things vLLM builds per request - `positions`, the attention metadata, and the paged KV cache block tables. The switch writes none of them; it builds constant Q/K/V and calls `vllm.Attention`. Section 5 of [`../../docs/MULTISWITCH_EXPLAINED.md`](../../docs/MULTISWITCH_EXPLAINED.md) walks that path in full.

Section 6 checks the observable consequence: a request's output should not depend on what it was batched with.

## 1 · Imports and configuration

In [ ]:
import json
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from transformers import AutoTokenizer

from granite_switch.config import GraniteSwitchConfig
from granite_switch.tutorials.vllm_server import (
    kill_stale_vllm_processes,
    launch_vllm,
    tail_log,
    wait_for_server,
)

BASE_MODEL = "ibm-granite/granite-4.1-3b"
GUARDIAN_LIB = "ibm-granite/granitelib-guardian-r1.0"
CORE_LIB = "ibm-granite/granitelib-core-r1.0"

# Reuse hello_multiswitch.ipynb's output if you have it; otherwise section 2 builds it.
MODEL_OUT = "./granite-switch-multi"
VLLM_PORT = 8000
VLLM_LOG = "./vllm_multiswitch.log"
BASE_URL = f"http://127.0.0.1:{VLLM_PORT}"

print(f"model: {MODEL_OUT}")
print(f"url:   {BASE_URL}")

## 2 · Compose the checkpoint

Skip this cell if `MODEL_OUT` already holds a `multi` checkpoint. `--switch-type multi` is the only line that differs from an ordinary build.

In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {GUARDIAN_LIB} {CORE_LIB} \
  --technology-filter lora \
  --switch-type multi \
  --output {MODEL_OUT}

In [ ]:
config = GraniteSwitchConfig.from_pretrained(MODEL_OUT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)

assert config.switch_type == "multi", (
    f"config.switch_type is {config.switch_type!r}; vLLM would build a SingleSwitch"
)

ADAPTER_A, ADAPTER_B = config.adapter_names[0], config.adapter_names[1]
CONTROL_A = tokenizer.convert_tokens_to_ids(f"<|{ADAPTER_A}|>")
CONTROL_B = tokenizer.convert_tokens_to_ids(f"<|{ADAPTER_B}|>")

print(f"switch_type : {config.switch_type}")
print(f"adapter 1   : {ADAPTER_A:<28} <|{ADAPTER_A}|> = {CONTROL_A}")
print(f"adapter 2   : {ADAPTER_B:<28} <|{ADAPTER_B}|> = {CONTROL_B}")

## 3 · Launch the server

Two settings matter more than the rest:

- **`max_num_seqs`** - the shipped helper defaults to `1`, which serializes everything. Section 6 is a batching test, so it has to be raised or the scheduler never puts two requests in one forward and the check silently proves nothing.
- **`max_model_len`** - kept modest so this fits a single small GPU.

There is deliberately **no** switch-related flag. If you find yourself looking for one, re-read **What changes when you serve it** above: the engine came from `config.json`.

In [ ]:
kill_stale_vllm_processes()  # a restarted notebook can leave a process holding the GPU

vllm_proc = launch_vllm(
    model=MODEL_OUT,
    port=VLLM_PORT,
    log_file=VLLM_LOG,
    max_num_seqs=8,  # must exceed 1 for section 6 to batch anything
    max_model_len=4096,
    enforce_eager=True,  # skips CUDA-graph capture; faster startup for a tutorial
)

if not wait_for_server(VLLM_PORT, log_file=VLLM_LOG):
    tail_log(VLLM_LOG, n=40)
    raise RuntimeError("vLLM did not come up - see the log tail above")

## 4 · Send a templated prompt as token ids

`/v1/completions` accepts a list of token ids in place of a string, which is what lets you control the exact token stream - including where control tokens sit.

The prompt still goes through `apply_chat_template`. That matters: the template supplies the role markers, the `<|end_of_text|>` turn terminator, and the `<|start_of_role|>assistant<|end_of_role|>` generation prompt. A bare control token followed by raw text omits all three, and the model continues the text instead of answering it.

`temperature=0.0` keeps generation greedy, which section 6 depends on - a sampled response would differ between runs for reasons that have nothing to do with batching.

In [ ]:
def complete(token_ids: list[int], max_tokens: int = 24) -> str:
    payload = {
        "model": MODEL_OUT,
        "prompt": token_ids,
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }
    request = urllib.request.Request(
        f"{BASE_URL}/v1/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        return json.loads(response.read().decode())["choices"][0]["text"]


ROLE_OPEN_MARKERS = ("<|start_of_role|>", "<|im_start|>")


def role_open_id() -> int:
    # Granite ships two template families - 4.0/4.1 role markers and 4.2 ChatML.
    for marker in ROLE_OPEN_MARKERS:
        token_id = tokenizer.convert_tokens_to_ids(marker)
        if token_id is not None and token_id != tokenizer.unk_token_id:
            return token_id
    raise RuntimeError(f"no role-open marker found among {ROLE_OPEN_MARKERS}")


def templated_ids(
    messages: list[dict],
    adapter_name: str | None = None,
    documents: list[dict] | None = None,
) -> list[int]:
    # Going through the chat template is what supplies the role markers, the turn
    # terminator, and the generation prompt. A bare control token followed by raw
    # text omits all three, and the model continues the text instead of answering.
    kwargs = {"adapter_name": adapter_name} if adapter_name else {}
    # documents= is what makes the prompt on-task for these adapters: both judge a
    # response AGAINST context, so with no documents there is nothing to judge.
    if documents is not None:
        kwargs["documents"] = documents
    out = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, **kwargs
    )
    # transformers 5.x returns a BatchEncoding here; 4.x returns a plain list.
    return list(out["input_ids"]) if hasattr(out, "keys") else list(out)


def build_multi_adapter_prompt(
    messages: list[dict],
    turn_adapter: str,
    generation_adapter: str,
    documents: list[dict] | None = None,
) -> list[int]:
    # The template binds ONE adapter (adapter_map is a dict, so a list raises
    # TypeError: unhashable type: 'list'). Template first, then splice the second
    # control token in before the generation prompt.
    ids = templated_ids(messages, turn_adapter, documents)
    cut = max(i for i, token_id in enumerate(ids) if token_id == role_open_id())
    control = tokenizer.convert_tokens_to_ids(f"<|{generation_adapter}|>")
    return ids[:cut] + [control] + ids[cut:]


# An ON-TASK prompt, and section 6 depends on it being one. Both guardian
# factuality adapters judge an assistant response against supplied context - their
# io.yaml declares outputs of {"score": "yes"|"no"} and {"correction": "..."} - so a
# bare "Summarize the meeting notes." gives them nothing to judge. Measured on a GPU
# with that prompt: the model was left ambivalent between answering as chat and
# emitting its verdict, 0.73 vs 0.27 on the very first token, and a bf16-sized
# numeric difference between the batched and unbatched paths flipped it in 8 of 10
# runs. Section 6 then reported "batching leaked" when nothing had leaked. A prompt
# the adapter is actually for moves the distribution off that boundary.
DOCUMENTS = [
    {
        "doc_id": "1",
        "text": (
            "The Q3 board meeting was held on 14 October. The board approved a 4% "
            "budget increase for the platform team and deferred the hiring plan to Q4."
        ),
    }
]
MESSAGES = [
    {
        "role": "user",
        "content": "When was the Q3 board meeting and what did it approve?",
    },
    # Deliberately wrong on both counts, so there is something for a factuality
    # adapter to find and the answer is not a coin flip.
    {
        "role": "assistant",
        "content": "It was held on 12 October and the board approved a 9% budget cut.",
    },
]

single_adapter = templated_ids(MESSAGES, ADAPTER_A, DOCUMENTS)
print(
    f"prompt ({len(single_adapter)} ids): {tokenizer.decode(single_adapter)[:200]!r} ..."
)
print(f"output: {complete(single_adapter)!r}")

## 5 · Two adapters in one served request

The same prompt shape as `hello_multiswitch.ipynb` section 5, now over HTTP. The server accepts it because control tokens are just ids in the stream - nothing on the serving path needs to know two adapters are involved.

In [ ]:
two_adapters = build_multi_adapter_prompt(MESSAGES, ADAPTER_A, ADAPTER_B, DOCUMENTS)

control_positions = [
    (i, tokenizer.convert_ids_to_tokens(t))
    for i, t in enumerate(two_adapters)
    if t in config.adapter_token_ids
]
print(f"prompt ({len(two_adapters)} ids): {tokenizer.decode(two_adapters)!r}")
print(f"control tokens at: {control_positions}")
print(f"output: {complete(two_adapters)!r}")

## 6 · Does batching leak between requests?

This is the question that decides whether you can deploy the thing. Three prompts naming different adapters are sent **twice**: once one at a time with the server otherwise idle, and once all at once so the scheduler batches them.

Greedy decoding makes each response a deterministic function of its own prompt. So if the isolated and concurrent responses match, the batch did not influence any individual request.

**A difference here is reported, not asserted, and that is deliberate.** Greedy decoding is deterministic in the prompt *mathematically*, not *bitwise*: a different batch shape can select different kernels and reduction orders, and where two candidate tokens are close the resulting shift flips the argmax - after which the whole continuation diverges. Measured on this checkpoint, first-token logprob gaps landed on an exact 0.25-nat grid (bf16 spacing), so a flip can turn on a single representable tick. An earlier version of this cell asserted equality and fired on exactly that near-tie, which made a notebook that was working look broken.

So the cell below prints a verdict and, when the two paths differ, tells you what to check. What *is* asserted is the property that cannot be explained away by numerics: the three adapters must not all return the same text. A uniform collapse to one adapter would make the isolation comparison pass for the wrong reason.

To chase a real difference: `granite-switch-internal/vela_yamls/ci/probe_first_token.py` prints the top-5 logprobs at token 0 for the batched and unbatched paths - a near-tie there is numerics, a confident disagreement is a leak - and `granite-switch-internal/vela_yamls/ci/diagnose_serving_leak.py` repeats the whole check with prefix caching off and on.

Note what this section does and does not establish. It checks the **observable** property over HTTP - identical output regardless of batch composition. It does not read per-token routing indices, which the server does not expose; `tests/vllm/test_multi_switch.py` covers that at the module level.


In [ ]:
PROMPTS = {
    f"A ({ADAPTER_A})": templated_ids(MESSAGES, ADAPTER_A, DOCUMENTS),
    f"B ({ADAPTER_B})": templated_ids(MESSAGES, ADAPTER_B, DOCUMENTS),
    "base (no adapter_name)": templated_ids(MESSAGES, documents=DOCUMENTS),
}

isolated = {name: complete(ids) for name, ids in PROMPTS.items()}

with ThreadPoolExecutor(max_workers=len(PROMPTS)) as pool:
    futures = {name: pool.submit(complete, ids) for name, ids in PROMPTS.items()}
    concurrent = {name: future.result() for name, future in futures.items()}

differed = [name for name in PROMPTS if isolated[name] != concurrent[name]]
total = len(PROMPTS)

print(f"{'request':<28} {'isolated == concurrent':<24} output (first 40 chars)")
for name in PROMPTS:
    same = isolated[name] == concurrent[name]
    print(f"{name:<28} {('yes' if same else 'NO'):<24} {isolated[name][:40]!r}")

print()
if not differed:
    # The count is load-bearing: granite-switch-internal/vela_yamls/ci/check_notebook_evidence.py only echoes a
    # --report line that contains a digit, so a digitless verdict never reaches the
    # job log and a green run would say nothing either way.
    print(f"no leak: {total} of {total} responses byte-identical under batching")
else:
    # Reported, not raised: a bf16 near-tie on any token flips the argmax and takes
    # the rest of the continuation with it, which is indistinguishable from a leak
    # at this level of observation. Escalate with the two granite-switch-internal/vela_yamls/ci probes.
    print(f"no leak verdict withheld: {len(differed)} of {total} responses differed.")
    print(f"the differing requests were {differed}.")
    print("This is NOT yet evidence of a leak. Check whether the first differing")
    print("token was a near-tie before concluding anything:")
    print(
        "  python granite-switch-internal/vela_yamls/ci/probe_first_token.py     # top-5 logprobs, both paths"
    )
    print(
        "  python granite-switch-internal/vela_yamls/ci/diagnose_serving_leak.py # repeat with the cache off/on"
    )
    for name in differed:
        print(f"\n  {name}\n    isolated   {isolated[name][:70]!r}")
        print(f"    concurrent {concurrent[name][:70]!r}")

The three adapters should also produce **different** output from each other. If they came back identical, routing collapsed to a single adapter and the isolation check above would pass for the wrong reason - a uniform failure looks exactly like perfect isolation.

In [ ]:
distinct = len(set(isolated.values()))
print(f"{distinct} distinct responses across {len(isolated)} adapters")
for name, text in isolated.items():
    print(f"  {name:<28} {text[:60]!r}")

assert distinct > 1, (
    "all adapters returned identical text - routing may have collapsed to one adapter, "
    "which would make the isolation check above meaningless"
)
print("routing did not collapse: the adapters produced distinct output")

## 7 · Serving flags worth knowing

**Prefix caching is on by default in vLLM V1**, and it interacts with adapter routing in a way that depends on your adapter technology.

- The e2e serving test passes `--no-enable-prefix-caching`, but that is a *measurement* requirement rather than a deployment one: the test replays identical prompts and needs real prefill forwards to trace, which the cache would skip.
- The deployment concern is different and specific to aLoRA. Because aLoRA places its control token *after* the conversation history, turn 2 shares a long prefix with turn 1 and will hit the cache - reusing KV entries computed while turn 1's adapter was active. Nothing in `src/granite_switch/` invalidates or salts that cache per adapter. `tests/integration/test_multi_switch_alora_cache.py` characterizes this deliberately: turn-2 routing correctness is asserted, while output-level contamination is *reported*, because whether an adapter's earlier output counts as legitimate context is a product call. [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) section 4 covers the placement difference behind it.

**Chunked prefill** splits a long prompt across forward passes, so a control token and the tokens it routes can land in different batches. This is exercised on purpose by the e2e test with `--enable-chunked-prefill` and a small `--max-num-batched-tokens`, and it works for the same reason decode does - the counting head reads `positions`, which vLLM keeps per request across chunks. See section 5 of [`../../docs/MULTISWITCH_EXPLAINED.md`](../../docs/MULTISWITCH_EXPLAINED.md).

**`enforce_eager=True`** is used above only to cut startup time for a tutorial. Drop it for real throughput measurements.

## 8 · Shut down

vLLM holds GPU memory until the process exits. Leaving it running is the usual cause of an out-of-memory error the next time a notebook tries to start a server.

In [ ]:
vllm_proc.terminate()
vllm_proc.wait(timeout=60)
print(f"server stopped (exit code {vllm_proc.returncode})")

## 9 · Next steps

- **Read the routing directly.** [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb) reads per-token adapter indices off a HuggingFace model, which the server does not expose, and measures where the default `single` engine breaks - CPU-only for that part, no download.
- **Drive a conversation over this server.** [`multi_turn_multiswitch.ipynb`](./multi_turn_multiswitch.ipynb) runs turn after turn against a vLLM server exactly like this one, and measures the prefix-cache reuse each KV policy buys.
- **Understand the batching path.** Section 5 of [`../../docs/MULTISWITCH_EXPLAINED.md`](../../docs/MULTISWITCH_EXPLAINED.md) covers the paged KV cache, per-request masking, decode carry, and chunked prefill.
- **Serve single-adapter workloads through mellea.** [`hello_mellea.ipynb`](./hello_mellea.ipynb) is the recommended path when one adapter per request is enough - it handles constrained decoding and output parsing for you.
- **Compare adapter technologies on throughput.** [`alora_vs_lora_race.ipynb`](./alora_vs_lora_race.ipynb) races LoRA against aLoRA on the same multi-step workload.
- **Compose your own checkpoint.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) is the full composer tour.
